In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('qoute_dataset.csv')

In [6]:
quotes = df['quote']

In [7]:
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [8]:
quotes = quotes.str.lower()

In [9]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [10]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
vocab_size = 10000

tokinizer = Tokenizer(num_words = vocab_size)
tokinizer.fit_on_texts(quotes)

In [13]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:20]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10),
 ('in', 11),
 ('be', 12),
 ('not', 13),
 ('are', 14),
 ('your', 15),
 ('have', 16),
 ('for', 17),
 ('but', 18),
 ('we', 19),
 ('if', 20)]

In [14]:
sequence = tokinizer.texts_to_sequences(quotes)

In [15]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [16]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [17]:
x = []
y =[]

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        x.append(input_seq)
        y.append(output_seq)

In [18]:
len(x)

85271

In [20]:
max_len = max(len(x) for x in x)
print(max_len)

745


In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(x, maxlen=max_len, padding='pre')


In [22]:
y = np.array(y)


In [23]:
X_padded.shape


(85271, 745)

In [24]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [25]:
y.shape


(85271,)

In [26]:
y_one_hot.shape


(85271, 10000)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [28]:
embedding_dim = 50
rnn_units = 128

In [29]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [30]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [31]:
rnn_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))


In [33]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [34]:
lstm_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
epochs = 10
batch_size = 128

In [37]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1

)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 52s 77ms/step - accuracy: 0.0437 - loss: 6.7474 - val_accuracy: 0.0537 - val_loss: 6.5617
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0573 - loss: 6.8220 - val_accuracy: 0.0643 - val_loss: 6.7685
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0754 - loss: 6.2998 - val_accuracy: 0.0802 - val_loss: 6.5805
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0901 - loss: 6.0724 - val_accuracy: 0.0888 - val_loss: 6.5436
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0981 - loss: 5.9151 - val_accuracy: 0.0966 - val_loss: 6.5132
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.1065 - loss: 5.7849 - val_accuracy: 0.0987 - val_loss: 6.5165
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.1148 - loss: 5.6579 - val_accuracy: 0.0983 - val_loss: 6.5347
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.1221 - loss: 5.5466 - 

In [38]:
epochs = 100
batch_size = 128
history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size= batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.0387 - loss: 6.7584 - val_accuracy: 0.0432 - val_loss: 6.6783
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0574 - loss: 6.3244 - val_accuracy: 0.0659 - val_loss: 6.5406
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0809 - loss: 6.0509 - val_accuracy: 0.0899 - val_loss: 6.4626
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0981 - loss: 5.8265 - val_accuracy: 0.0955 - val_loss: 6.4382
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1086 - loss: 5.6425 - val_accuracy: 0.0970 - val_loss: 6.4237
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1192 - loss: 5.4700 - val_accuracy: 0.1062 - val_loss: 6.4266
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1289 - loss: 5.3094 - val_accuracy: 0.1080 - val_loss: 6.4490
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1379 - loss: 5

In [39]:
from tensorflow.keras.models import load_model


In [42]:
lstm_model = load_model("lstm_model.h5")


In [41]:
lstm_model.save("lstm_model.h5")


In [44]:
index_to_word ={}
for word,index in word_index.items():
  index_to_word[index] = word

In [45]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [53]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [54]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

going


In [55]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text


In [56]:
seed = 'are you a'
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a great and if you are going to try to make


In [57]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)



In [58]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)